# 03 OCR - 00: Pretrained Gate (Pilot)

Pilot gate for pretrained OCR candidates on `wm_polygon` and `wm_bbox` crops.
Candidates are included into comparison CSV only if Stage A conditions pass.

In [71]:
import sys, json, time
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/WaterMeterCV")
    DATA_ROOT = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    RESULTS_DIR = Path("/content/drive/MyDrive/WaterMeterCV/results")
else:
    ROOT = Path("../..").resolve()
    DATA_ROOT = ROOT / "WaterMetricsDATA"
    RESULTS_DIR = ROOT / "results"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib
import models.data.ocr_dataset as _ocr_dataset
importlib.reload(_ocr_dataset)
prepare_ocr_crops = _ocr_dataset.prepare_ocr_crops
load_ocr_crops = _ocr_dataset.load_ocr_crops
from models.metrics.ocr_metrics import evaluate_ocr_batch, build_ocr_comparison_row, append_ocr_comparison_row
from models.utils.orientation import dual_read_inference

CROPS_ROOT = DATA_ROOT / "ocr_crops"
WM_PATH = DATA_ROOT / "waterMeterDataset/WaterMeters"
WM_ROI_YOLO_WEIGHTS = ROOT / "models/weights/roi_yolo/wm_yolo_roi_20260412_230832/weights/best.pt"
WM_BBOX_SOURCE = "yolo"  # "yolo" or "gt"
WM_LABEL_MODE = "wm_fraction_aware"  # "source" or "wm_fraction_aware"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def build_wm_roi_detector(weights_path: Path, source: str = WM_BBOX_SOURCE):
    if source != "yolo":
        return None
    if not weights_path.exists():
        print(f"[prepare_ocr_crops] YOLO ROI weights not found: {weights_path}. Fallback to GT bbox.")
        return None

    try:
        from ultralytics import YOLO
        model = YOLO(str(weights_path))
    except Exception as exc:
        print(f"[prepare_ocr_crops] Failed to load YOLO ROI model: {exc}. Fallback to GT bbox.")
        return None

    def _detect(img_bgr):
        try:
            result = model.predict(img_bgr, verbose=False, conf=0.001, imgsz=640, max_det=16)[0]
        except Exception:
            return None

        boxes = getattr(result, "boxes", None)
        if boxes is None or len(boxes) == 0:
            return None

        try:
            idx = int(boxes.conf.argmax().item()) if hasattr(boxes, "conf") else 0
        except Exception:
            idx = 0

        b = boxes.xywhn[idx]
        return float(b[0]), float(b[1]), float(b[2]), float(b[3])

    return _detect


wm_roi_detector = build_wm_roi_detector(WM_ROI_YOLO_WEIGHTS, source=WM_BBOX_SOURCE)
prepare_ocr_crops(
    WM_PATH,
    CROPS_ROOT,
    roi_detector=wm_roi_detector,
    fallback_to_gt_on_miss=True,
    label_mode=WM_LABEL_MODE,
)
print("wm_bbox source:", "yolo+gt_fallback" if wm_roi_detector is not None else "gt")
print("wm label mode:", WM_LABEL_MODE)
wm_poly_test = load_ocr_crops(CROPS_ROOT / "wm_polygon", "test")
wm_bbox_test = load_ocr_crops(CROPS_ROOT / "wm_bbox", "test")

# Toggle preprocessing strategy for OCR engines.
# Options: "none", "gray_clahe"
PREPROCESS_MODE = "gray_clahe"
MAX_READING_DIGITS = 10
LEADING_ZERO_ORIENTATION_MIN = 3
REGISTRY_EXCLUDE_CANDIDATES = {"doctr", "easyocr"}

# Ultralytics heuristics
ULTRA_LAST_DRUM_X_ALIGN_FACTOR = 0.55
ULTRA_LAST_DRUM_MIN_Y_GAP_FACTOR = 0.25
ULTRA_OVERLAP_IOA_MIN = 0.65
ULTRA_OVERLAP_CENTER_FACTOR = 0.55

# Red-cluster orientation heuristic (computed only inside Ultralytics bboxes)
RED_CLUSTER_MIN_COVERAGE = 0.0025
RED_CLUSTER_LEFT_MAX_X = 0.43
RED_CLUSTER_RIGHT_MIN_X = 0.57
RED_BBOX_MIN_ACTIVE_PIXELS = 64

# Statistical tail heuristic
LONG_TAIL_ZERO_MIN_DIGITS = 8
LONG_TAIL_ZERO_MIN_SUFFIX = 2

# Orientation voting weights (sum ~= 1.0)
RED_BBOX_VOTE_WEIGHT = 0.85
STAT_TAIL_VOTE_WEIGHT = 0.09
LEADING_ZERO_VOTE_WEIGHT = 0.06

# Make red-box prior slightly stricter (~8%).
RED_BBOX_STRICTNESS = 0.08

ULTRALYTICS_MODEL_CACHE = {}


def preprocess_for_ocr(image_bgr, mode=PREPROCESS_MODE):
    if mode == "none":
        return image_bgr

    if mode == "gray_clahe":
        gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

    raise ValueError(f"Unsupported preprocess mode: {mode!r}")


def wrap_predictor_with_preprocess(predictor, mode=PREPROCESS_MODE):
    def _predict(image_bgr):
        preprocessed = preprocess_for_ocr(image_bgr, mode=mode)
        return predictor(preprocessed)

    return _predict


def digits_only(text):
    return "".join(ch for ch in str(text) if ch.isdigit())


def safe_mean(values):
    return float(np.mean(values)) if values else 0.0


def apply_max_digits_heuristic(pred, conf, max_digits=MAX_READING_DIGITS):
    pred_digits = digits_only(pred)
    try:
        conf_val = float(conf)
    except Exception:
        conf_val = 0.0

    if len(pred_digits) <= max_digits:
        return pred_digits, conf_val

    # Penalize overlong outputs and keep only the leftmost max_digits.
    overflow = len(pred_digits) - max_digits
    penalty = max(0.0, 1.0 - (overflow / max_digits))
    return pred_digits[:max_digits], conf_val * penalty


def wrap_predictor_with_max_digits(predictor, max_digits=MAX_READING_DIGITS):
    def _predict(image_bgr):
        pred, conf = predictor(image_bgr)
        return apply_max_digits_heuristic(pred, conf, max_digits=max_digits)

    return _predict


def leading_zero_count(text):
    d = digits_only(text)
    n = 0
    for ch in d:
        if ch == "0":
            n += 1
        else:
            break
    return n


def normalize_digits_for_stats(text):
    d = digits_only(text).lstrip("0")
    return d if d else "0"


def trailing_zero_count(text):
    n = 0
    for ch in reversed(str(text)):
        if ch == "0":
            n += 1
        else:
            break
    return n


def is_long_tail_zero_pattern(text):
    norm = normalize_digits_for_stats(text)
    return len(norm) >= LONG_TAIL_ZERO_MIN_DIGITS and trailing_zero_count(norm) >= LONG_TAIL_ZERO_MIN_SUFFIX


def ultralytics_digit_rank(digit):
    d = int(digit)
    # Drum rollover rule: 0 is treated as greater than 9.
    return 10 if d == 0 else d


def bbox_intersection_area(a, b):
    x_left = max(a["x1"], b["x1"])
    y_top = max(a["y1"], b["y1"])
    x_right = min(a["x2"], b["x2"])
    y_bottom = min(a["y2"], b["y2"])
    if x_right <= x_left or y_bottom <= y_top:
        return 0.0
    return float((x_right - x_left) * (y_bottom - y_top))


def bbox_area(d):
    return float(max(d["x2"] - d["x1"], 0.0) * max(d["y2"] - d["y1"], 0.0))


def boxes_are_nested_or_almost_nested(a, b):
    inter = bbox_intersection_area(a, b)
    if inter <= 0.0:
        return False

    area_a = max(bbox_area(a), 1e-6)
    area_b = max(bbox_area(b), 1e-6)
    ioa_small = inter / min(area_a, area_b)

    if ioa_small < ULTRA_OVERLAP_IOA_MIN:
        return False

    x_close = abs(a["cx"] - b["cx"]) <= ULTRA_OVERLAP_CENTER_FACTOR * max(a["w"], b["w"], 1e-6)
    y_close = abs(a["cy"] - b["cy"]) <= ULTRA_OVERLAP_CENTER_FACTOR * max(a["h"], b["h"], 1e-6)
    return bool(x_close and y_close)


def apply_ultralytics_overlap_heuristic(detections):
    dets = sorted(detections, key=lambda d: d["cx"])
    n = len(dets)
    if n < 2:
        return dets, {"applied": False, "reason": "not_enough_boxes"}

    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a_idx, b_idx):
        ra, rb = find(a_idx), find(b_idx)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if boxes_are_nested_or_almost_nested(dets[i], dets[j]):
                union(i, j)

    groups = {}
    for idx in range(n):
        root = find(idx)
        groups.setdefault(root, []).append(dets[idx])

    filtered = []
    applied_count = 0
    for _, group in groups.items():
        if len(group) == 1:
            filtered.append(group[0])
            continue

        applied_count += 1
        zeros = [g for g in group if int(g["digit"]) == 0]
        if zeros:
            chosen = max(zeros, key=lambda g: (float(g["conf"]), -bbox_area(g)))
        else:
            chosen = max(group, key=lambda g: (ultralytics_digit_rank(g["digit"]), float(g["conf"])))
        filtered.append(chosen)

    filtered.sort(key=lambda d: d["cx"])
    return filtered, {
        "applied": applied_count > 0,
        "reason": "overlap_group",
        "group_count": int(applied_count),
    }


def apply_ultralytics_last_drum_heuristic(detections):
    dets = sorted(detections, key=lambda d: d["cx"])
    if len(dets) < 2:
        return dets, {"applied": False, "reason": "not_enough_boxes"}

    a, b = dets[-2], dets[-1]
    w_ref = max(a["w"], b["w"], 1e-6)
    h_ref = max(a["h"], b["h"], 1e-6)

    x_close = abs(a["cx"] - b["cx"]) <= ULTRA_LAST_DRUM_X_ALIGN_FACTOR * w_ref
    y_split = abs(a["cy"] - b["cy"]) >= ULTRA_LAST_DRUM_MIN_Y_GAP_FACTOR * h_ref
    x_overlap = min(a["x2"], b["x2"]) > max(a["x1"], b["x1"])

    if not (x_close and y_split and x_overlap):
        return dets, {"applied": False, "reason": "not_vertical_pair"}

    # Prefer stronger digit first (with 0 > 9), then confidence.
    rank_a = ultralytics_digit_rank(a["digit"])
    rank_b = ultralytics_digit_rank(b["digit"])
    if rank_a != rank_b:
        chosen = a if rank_a > rank_b else b
        reason = "digit_rank"
    elif a["conf"] != b["conf"]:
        chosen = a if a["conf"] >= b["conf"] else b
        reason = "confidence"
    elif a["cy"] != b["cy"]:
        chosen = a if a["cy"] < b["cy"] else b
        reason = "upper_tiebreak"
    else:
        chosen = a
        reason = "left_tiebreak"

    dropped = b if chosen is a else a
    filtered = dets[:-2] + [chosen]
    return filtered, {
        "applied": True,
        "reason": reason,
        "kept_digit": int(chosen["digit"]),
        "dropped_digit": int(dropped["digit"]),
    }


def get_ultralytics_baseline_model():
    cache_key = "ultralytics_yolo11m_baseline"
    if cache_key in ULTRALYTICS_MODEL_CACHE:
        return ULTRALYTICS_MODEL_CACHE[cache_key]

    try:
        from ultralytics import YOLO

        yolo_run_dir = ROOT / "models/weights/baseline_yolo/yolo11m_20260414_194809"
        yolo_weight_candidates = [
            yolo_run_dir / "weights/best.pt",
            yolo_run_dir / "weights/last.pt",
            yolo_run_dir if yolo_run_dir.suffix.lower() == ".pt" else None,
        ]
        yolo_weight_path = next((p for p in yolo_weight_candidates if p is not None and p.exists()), None)
        if yolo_weight_path is None:
            ULTRALYTICS_MODEL_CACHE[cache_key] = None
            return None

        model = YOLO(str(yolo_weight_path))
        ULTRALYTICS_MODEL_CACHE[cache_key] = model
        return model
    except Exception:
        ULTRALYTICS_MODEL_CACHE[cache_key] = None
        return None


def extract_ultralytics_digit_detections(image_bgr, model=None):
    if image_bgr is None:
        return []

    if model is None:
        model = get_ultralytics_baseline_model()
    if model is None:
        return []

    try:
        result = model.predict(image_bgr, verbose=False, imgsz=320, max_det=32)[0]
    except Exception:
        return []

    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        return []

    digit_mask = boxes.cls <= 9
    digit_boxes = boxes[digit_mask]
    if len(digit_boxes) == 0:
        return []

    sorted_idx = digit_boxes.xywh[:, 0].argsort()
    detections = []
    for i in sorted_idx:
        xyxy = digit_boxes.xyxy[i].tolist()
        xywh = digit_boxes.xywh[i].tolist()
        if len(xyxy) < 4 or len(xywh) < 4:
            continue

        x1, y1, x2, y2 = [float(v) for v in xyxy[:4]]
        cx, cy, w, h = [float(v) for v in xywh[:4]]
        digit = int(digit_boxes.cls[i].item())
        score = float(digit_boxes.conf[i].item()) if hasattr(digit_boxes, "conf") else 0.0

        detections.append(
            {
                "digit": digit,
                "conf": score,
                "score": score,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "cx": cx,
                "cy": cy,
                "w": w,
                "h": h,
            }
        )

    detections.sort(key=lambda d: d["cx"])
    return detections


def estimate_red_horizontal_cluster_in_bboxes(
    image_bgr,
    detections,
    min_coverage=RED_CLUSTER_MIN_COVERAGE,
    min_active_pixels=RED_BBOX_MIN_ACTIVE_PIXELS,
):
    if image_bgr is None or image_bgr.size == 0 or not detections:
        return None

    b, g, r = cv2.split(image_bgr.astype(np.float32))
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    s = hsv[:, :, 1].astype(np.float32)
    v = hsv[:, :, 2].astype(np.float32)
    red_score = (r - np.maximum(g, b)) + 0.25 * s

    h, w = image_bgr.shape[:2]
    weighted_x_sum = 0.0
    weight_total = 0.0
    active_pixels = 0

    for d in detections:
        x1 = max(0, min(w - 1, int(round(d["x1"]))))
        y1 = max(0, min(h - 1, int(round(d["y1"]))))
        x2 = max(0, min(w - 1, int(round(d["x2"]))))
        y2 = max(0, min(h - 1, int(round(d["y2"]))))
        if x2 <= x1 or y2 <= y1:
            continue

        rr = r[y1 : y2 + 1, x1 : x2 + 1]
        gg = g[y1 : y2 + 1, x1 : x2 + 1]
        bb = b[y1 : y2 + 1, x1 : x2 + 1]
        vv = v[y1 : y2 + 1, x1 : x2 + 1]
        score_patch = red_score[y1 : y2 + 1, x1 : x2 + 1]

        base_mask = (vv >= 30.0) & (rr >= 40.0) & (rr >= gg * 1.02) & (rr >= bb * 1.02)
        if not np.any(base_mask):
            continue

        vals = score_patch[base_mask]
        thr = max(float(np.percentile(vals, 85.0)), 10.0)
        strong = base_mask & (score_patch >= thr)
        if not np.any(strong):
            continue

        _, xs = np.where(strong)
        weights = np.maximum(score_patch[strong] - thr, 0.0) + 1e-3
        weighted_x_sum += float(np.sum((xs.astype(np.float32) + float(x1)) * weights))
        weight_total += float(np.sum(weights))
        active_pixels += int(np.sum(strong))

    if active_pixels < int(min_active_pixels) or weight_total <= 1e-6:
        return None

    denom = float(max(w - 1, 1))
    x_norm = float(weighted_x_sum / (weight_total * denom))
    coverage = float(active_pixels / float(max(h * w, 1)))
    if coverage < float(min_coverage):
        return None

    return {"x_norm": x_norm, "coverage": coverage, "active_pixels": int(active_pixels)}


def get_stricter_red_bbox_thresholds(strictness=RED_BBOX_STRICTNESS):
    center = 0.5
    s = float(np.clip(strictness, 0.0, 0.30))
    left_dist = max(center - RED_CLUSTER_LEFT_MAX_X, 0.0)
    right_dist = max(RED_CLUSTER_RIGHT_MIN_X - center, 0.0)

    left_thr = RED_CLUSTER_LEFT_MAX_X - left_dist * s
    right_thr = RED_CLUSTER_RIGHT_MIN_X + right_dist * s
    min_cov = RED_CLUSTER_MIN_COVERAGE * (1.0 + s)
    min_pixels = int(np.ceil(RED_BBOX_MIN_ACTIVE_PIXELS * (1.0 + s)))
    return left_thr, right_thr, min_cov, min_pixels


def red_bbox_orientation_prior(image_bgr):
    detections = extract_ultralytics_digit_detections(image_bgr)
    if not detections:
        return None

    detections, _ = apply_ultralytics_overlap_heuristic(detections)
    detections, _ = apply_ultralytics_last_drum_heuristic(detections)

    left_thr, right_thr, min_cov, min_pixels = get_stricter_red_bbox_thresholds()
    stats = estimate_red_horizontal_cluster_in_bboxes(
        image_bgr,
        detections,
        min_coverage=min_cov,
        min_active_pixels=min_pixels,
    )
    if stats is None:
        return None

    x_norm = stats["x_norm"]
    base = {
        **stats,
        "left_thr": float(left_thr),
        "right_thr": float(right_thr),
        "strict_min_coverage": float(min_cov),
        "strict_min_pixels": int(min_pixels),
    }
    if x_norm >= right_thr:
        return {**base, "angle": 0, "reason": "red_bbox_right"}
    if x_norm <= left_thr:
        return {**base, "angle": 180, "reason": "red_bbox_left"}
    return None


def _register_orientation_vote(votes, vote_details, name, angle, weight, extra=None):
    if angle not in (0, 180):
        return
    weight = float(max(weight, 0.0))
    if weight <= 0.0:
        return

    a = int(angle)
    votes[a] = float(votes.get(a, 0.0) + weight)
    detail = {"angle": a, "weight": weight}
    if isinstance(extra, dict):
        detail.update(extra)
    vote_details[name] = detail


def select_dual_orientation_with_priors(dual, image_bgr=None, min_leading_zeros=LEADING_ZERO_ORIENTATION_MIN):
    z0 = leading_zero_count(dual.pred_0)
    z180 = leading_zero_count(dual.pred_180)
    conf0 = float(dual.conf_0)
    conf180 = float(dual.conf_180)

    votes = {0: 0.0, 180: 0.0}
    vote_details = {}

    red_prior = red_bbox_orientation_prior(image_bgr) if image_bgr is not None else None
    if red_prior is not None:
        _register_orientation_vote(
            votes,
            vote_details,
            "red_bbox",
            int(red_prior["angle"]),
            RED_BBOX_VOTE_WEIGHT,
            {
                "reason": red_prior.get("reason"),
                "x_norm": float(red_prior.get("x_norm", 0.0)),
                "coverage": float(red_prior.get("coverage", 0.0)),
            },
        )

    tail0 = bool(is_long_tail_zero_pattern(dual.pred_0))
    tail180 = bool(is_long_tail_zero_pattern(dual.pred_180))
    if tail0 and not tail180:
        _register_orientation_vote(
            votes,
            vote_details,
            "long_tail_zero",
            180,
            STAT_TAIL_VOTE_WEIGHT,
            {"tail0": True, "tail180": False},
        )
    elif tail180 and not tail0:
        _register_orientation_vote(
            votes,
            vote_details,
            "long_tail_zero",
            0,
            STAT_TAIL_VOTE_WEIGHT,
            {"tail0": False, "tail180": True},
        )

    if z0 >= min_leading_zeros and z180 < min_leading_zeros:
        _register_orientation_vote(
            votes,
            vote_details,
            "leading_zeros",
            0,
            LEADING_ZERO_VOTE_WEIGHT,
            {"leading_zeros_0": int(z0), "leading_zeros_180": int(z180)},
        )
    elif z180 >= min_leading_zeros and z0 < min_leading_zeros:
        _register_orientation_vote(
            votes,
            vote_details,
            "leading_zeros",
            180,
            LEADING_ZERO_VOTE_WEIGHT,
            {"leading_zeros_0": int(z0), "leading_zeros_180": int(z180)},
        )

    vote_score_0 = float(votes[0])
    vote_score_180 = float(votes[180])
    if vote_score_180 > vote_score_0 + 1e-9:
        selected_angle = 180
        reason = "vote_180"
    elif vote_score_0 > vote_score_180 + 1e-9:
        selected_angle = 0
        reason = "vote_0"
    elif conf180 > conf0:
        selected_angle = 180
        reason = "confidence_180_tiebreak"
    else:
        selected_angle = 0
        reason = "confidence_0_tiebreak"

    selected_pred = dual.pred_180 if selected_angle == 180 else dual.pred_0
    selected_conf = conf180 if selected_angle == 180 else conf0

    selected_vote_contrib = {}
    for h_name, h_data in vote_details.items():
        h_angle = int(h_data.get("angle", -1))
        h_weight = float(h_data.get("weight", 0.0))
        selected_vote_contrib[h_name] = h_weight if h_angle == selected_angle else 0.0

    result = {
        "selected_pred": selected_pred,
        "selected_conf": float(selected_conf),
        "selected_angle": int(selected_angle),
        "reason": reason,
        "leading_zeros_0": int(z0),
        "leading_zeros_180": int(z180),
        "conf_0": conf0,
        "conf_180": conf180,
        "vote_score_0": vote_score_0,
        "vote_score_180": vote_score_180,
        "vote_margin": float(abs(vote_score_0 - vote_score_180)),
        "vote_total": float(vote_score_0 + vote_score_180),
        "heuristic_votes": vote_details,
        "selected_vote_contrib": selected_vote_contrib,
    }

    if red_prior is not None:
        result["red_bbox_x_norm"] = float(red_prior.get("x_norm", 0.0))
        result["red_bbox_coverage"] = float(red_prior.get("coverage", 0.0))
        result["red_bbox_left_thr"] = float(red_prior.get("left_thr", RED_CLUSTER_LEFT_MAX_X))
        result["red_bbox_right_thr"] = float(red_prior.get("right_thr", RED_CLUSTER_RIGHT_MIN_X))

    return result


def build_registry():
    reg, errs = {}, {}

    if "easyocr" not in REGISTRY_EXCLUDE_CANDIDATES:
        try:
            import easyocr

            reader = easyocr.Reader(["en"], gpu=False)

            def pred_easy(img):
                parts = []
                for box, txt, conf in reader.readtext(
                    img, detail=1, paragraph=False, allowlist="0123456789"
                ):
                    d = digits_only(txt)
                    if d:
                        left = min(float(p[0]) for p in box)
                        parts.append((left, d, float(conf)))
                if not parts:
                    return "", 0.0
                parts.sort(key=lambda x: x[0])
                return "".join(x[1] for x in parts), safe_mean([x[2] for x in parts])

            reg["easyocr"] = pred_easy
        except Exception as exc:
            errs["easyocr"] = str(exc)

    try:
        import os

        os.environ["FLAGS_use_mkldnn"] = "0"

        from paddleocr import PaddleOCR

        paddle_kwargs = {
            "use_doc_orientation_classify": False,
            "use_doc_unwarping": False,
            "use_textline_orientation": True,
            "lang": "en",
        }
        try:
            ocr = PaddleOCR(show_log=False, **paddle_kwargs)
        except Exception:
            ocr = PaddleOCR(**paddle_kwargs)

        # Early runtime sanity check (Windows + oneDNN can fail at predict time).
        try:
            _ = ocr.predict(np.zeros((64, 256, 3), dtype=np.uint8))
        except Exception as exc:
            errs["paddleocr"] = f"runtime predict failed: {type(exc).__name__}: {exc}"
        else:

            def _append_part(parts, box, txt, conf, idx_fallback):
                d = digits_only(txt)
                if not d:
                    return

                left = float(idx_fallback)
                if box is not None:
                    arr = np.asarray(box, dtype=np.float32)
                    if arr.ndim == 2 and arr.shape[1] >= 2:
                        left = float(np.min(arr[:, 0]))
                parts.append((left, d, float(conf)))

            def _parse_predict_results(out):
                parts = []
                for item_idx, item in enumerate(out or []):
                    if item is None:
                        continue

                    payload = None
                    if hasattr(item, "to_dict"):
                        try:
                            payload = item.to_dict()
                        except Exception:
                            payload = None
                    elif isinstance(item, dict):
                        payload = item

                    if payload is not None:
                        texts = payload.get("rec_texts") or payload.get("texts") or []
                        scores = payload.get("rec_scores") or payload.get("scores") or []
                        boxes = payload.get("rec_boxes") or payload.get("dt_polys") or payload.get("boxes") or []

                        for i, txt in enumerate(texts):
                            score = float(scores[i]) if i < len(scores) else 0.0
                            box = boxes[i] if i < len(boxes) else None
                            _append_part(parts, box, txt, score, idx_fallback=i)
                        continue

                    if isinstance(item, list):
                        for i, entry in enumerate(item):
                            if entry is None or len(entry) < 2:
                                continue
                            box = entry[0]
                            rec = entry[1]
                            if isinstance(rec, (list, tuple)):
                                txt = rec[0] if len(rec) > 0 else ""
                                conf = float(rec[1]) if len(rec) > 1 else 0.0
                            else:
                                txt = str(rec)
                                conf = 0.0
                            _append_part(parts, box, txt, conf, idx_fallback=i + item_idx * 1000)

                    elif isinstance(item, tuple) and len(item) == 2:
                        box, rec = item
                        if isinstance(rec, (list, tuple)):
                            txt = rec[0] if len(rec) > 0 else ""
                            conf = float(rec[1]) if len(rec) > 1 else 0.0
                        else:
                            txt = str(rec)
                            conf = 0.0
                        _append_part(parts, box, txt, conf, idx_fallback=item_idx)

                return parts

            def pred_paddle(img):
                try:
                    out = ocr.predict(img)
                except Exception:
                    return "", 0.0

                parts = _parse_predict_results(out)
                if not parts:
                    return "", 0.0
                parts.sort(key=lambda x: x[0])
                return "".join(x[1] for x in parts), safe_mean([x[2] for x in parts])

            reg["paddleocr"] = pred_paddle
    except Exception as exc:
        errs["paddleocr"] = str(exc)

    try:
        import pytesseract
        from pytesseract import Output

        _ = pytesseract.get_tesseract_version()

        def pred_tesseract(img):
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            cfg = "--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789"
            try:
                data = pytesseract.image_to_data(gray, config=cfg, output_type=Output.DICT)
            except Exception:
                return "", 0.0

            parts = []
            count = len(data.get("text", []))
            lefts = data.get("left", [])
            confs = data.get("conf", [])
            texts = data.get("text", [])

            for i in range(count):
                txt = texts[i] if i < len(texts) else ""
                d = digits_only(txt)
                if not d:
                    continue

                left = float(lefts[i]) if i < len(lefts) else float(i)
                conf_raw = confs[i] if i < len(confs) else 0.0
                try:
                    conf = max(0.0, float(conf_raw))
                except Exception:
                    conf = 0.0
                if conf > 1.0:
                    conf /= 100.0
                parts.append((left, d, conf))

            if not parts:
                txt = pytesseract.image_to_string(gray, config=cfg)
                d = digits_only(txt)
                if d:
                    return d, 0.0
                return "", 0.0

            parts.sort(key=lambda x: x[0])
            return "".join(x[1] for x in parts), safe_mean([x[2] for x in parts])

        reg["pytesseract"] = pred_tesseract
    except Exception as exc:
        errs["pytesseract"] = str(exc)

    if "doctr" not in REGISTRY_EXCLUDE_CANDIDATES:
        try:
            from doctr.models import ocr_predictor

            doctr_model = ocr_predictor(pretrained=True)

            def pred_doctr(img):
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                try:
                    exported = doctr_model([rgb]).export()
                except Exception:
                    return "", 0.0

                parts = []
                try:
                    pages = exported.get("pages", [])
                    if not pages:
                        return "", 0.0

                    for block_idx, block in enumerate(pages[0].get("blocks", [])):
                        for line_idx, line in enumerate(block.get("lines", [])):
                            for word_idx, word in enumerate(line.get("words", [])):
                                d = digits_only(word.get("value", ""))
                                if not d:
                                    continue

                                left = float(block_idx * 10_000 + line_idx * 1_000 + word_idx)
                                geom = word.get("geometry")
                                if (
                                    isinstance(geom, (list, tuple))
                                    and len(geom) >= 1
                                    and isinstance(geom[0], (list, tuple))
                                    and len(geom[0]) >= 1
                                ):
                                    left = float(geom[0][0])

                                conf = float(word.get("confidence") or 0.0)
                                parts.append((left, d, conf))
                except Exception:
                    return "", 0.0

                if not parts:
                    return "", 0.0
                parts.sort(key=lambda x: x[0])
                return "".join(x[1] for x in parts), safe_mean([x[2] for x in parts])

            reg["doctr"] = pred_doctr
        except Exception as exc:
            errs["doctr"] = str(exc)

    try:
        yolo_model = get_ultralytics_baseline_model()
        if yolo_model is None:
            raise FileNotFoundError("YOLO weights not found for ultralytics_yolo11m_baseline")

        # Runtime sanity check to fail fast if model cannot infer in current environment.
        _ = yolo_model.predict(np.zeros((64, 256, 3), dtype=np.uint8), verbose=False, imgsz=320, max_det=32)

        def pred_ultralytics_yolo_baseline(img):
            detections = extract_ultralytics_digit_detections(img, model=yolo_model)
            if not detections:
                return "", 0.0

            detections, _ = apply_ultralytics_overlap_heuristic(detections)
            detections, _ = apply_ultralytics_last_drum_heuristic(detections)
            if not detections:
                return "", 0.0

            pred_str = "".join(str(int(d["digit"])) for d in detections)
            confs = [float(d["conf"]) for d in detections]
            return pred_str, safe_mean(confs)

        reg["ultralytics_yolo11m_baseline"] = pred_ultralytics_yolo_baseline
    except Exception as exc:
        errs["ultralytics_yolo11m_baseline"] = str(exc)

    return reg, errs


print(f"Preprocess mode: {PREPROCESS_MODE}")

wm_bbox source: yolo+gt_fallback
wm label mode: wm_fraction_aware
Preprocess mode: gray_clahe


In [66]:
def evaluate_predictor(predictor, samples):
    preds, gts, times = [], [], []
    non_empty, chosen_180 = 0, 0
    leading_zero_overrides = 0
    red_bbox_overrides = 0
    long_tail_zero_overrides = 0
    confidence_tiebreaks = 0

    for img_path, gt in samples:
        image = cv2.imread(str(img_path))
        if image is None:
            continue

        t0 = time.perf_counter()
        dual = dual_read_inference(image, predictor)
        selected = select_dual_orientation_with_priors(dual, image_bgr=image)

        times.append((time.perf_counter() - t0) * 1000.0)
        preds.append(selected["selected_pred"])
        gts.append(gt)

        non_empty += int(selected["selected_pred"] != "")
        chosen_180 += int(selected["selected_angle"] == 180)

        contrib = selected.get("selected_vote_contrib", {})
        leading_zero_overrides += int(float(contrib.get("leading_zeros", 0.0)) > 0.0)
        red_bbox_overrides += int(float(contrib.get("red_bbox", 0.0)) > 0.0)
        long_tail_zero_overrides += int(float(contrib.get("long_tail_zero", 0.0)) > 0.0)
        confidence_tiebreaks += int(str(selected.get("reason", "")).startswith("confidence_"))

    m = evaluate_ocr_batch(preds, gts) if gts else {"fsa_raw": 0.0, "fsa_norm": 0.0, "pda": 0.0, "cer": 1.0}
    return {
        **m,
        "avg_ms": safe_mean(times),
        "non_empty_rate": float(non_empty / max(len(gts), 1)),
        "selected_180_share": float(chosen_180 / max(len(gts), 1)),
        "leading_zero_override_rate": float(leading_zero_overrides / max(len(gts), 1)),
        "red_bbox_override_rate": float(red_bbox_overrides / max(len(gts), 1)),
        "long_tail_zero_override_rate": float(long_tail_zero_overrides / max(len(gts), 1)),
        "confidence_tiebreak_rate": float(confidence_tiebreaks / max(len(gts), 1)),
    }


registry, init_errors = build_registry()
MIN_NON_EMPTY_RATE = 0.10
MAX_AVG_MS = 1500.0
NO_PREPROCESS_CANDIDATES = {"ultralytics_yolo11m_baseline"}
EVAL_EXCLUDE_CANDIDATES = {"doctr", "easyocr"}

strict_left_thr, strict_right_thr, strict_min_cov, strict_min_pixels = get_stricter_red_bbox_thresholds()

rows = []
report = {
    "run_date": datetime.now().isoformat(),
    "preprocess_mode": PREPROCESS_MODE,
    "max_reading_digits": MAX_READING_DIGITS,
    "leading_zero_orientation_min": LEADING_ZERO_ORIENTATION_MIN,
    "long_tail_zero_min_digits": LONG_TAIL_ZERO_MIN_DIGITS,
    "long_tail_zero_min_suffix": LONG_TAIL_ZERO_MIN_SUFFIX,
    "red_cluster_min_coverage": RED_CLUSTER_MIN_COVERAGE,
    "red_cluster_left_max_x": RED_CLUSTER_LEFT_MAX_X,
    "red_cluster_right_min_x": RED_CLUSTER_RIGHT_MIN_X,
    "red_bbox_min_active_pixels": RED_BBOX_MIN_ACTIVE_PIXELS,
    "red_bbox_vote_weight": RED_BBOX_VOTE_WEIGHT,
    "stat_tail_vote_weight": STAT_TAIL_VOTE_WEIGHT,
    "leading_zero_vote_weight": LEADING_ZERO_VOTE_WEIGHT,
    "red_bbox_strictness": RED_BBOX_STRICTNESS,
    "red_bbox_strict_left_max_x": float(strict_left_thr),
    "red_bbox_strict_right_min_x": float(strict_right_thr),
    "red_bbox_strict_min_coverage": float(strict_min_cov),
    "red_bbox_strict_min_pixels": int(strict_min_pixels),
    "eval_exclude_candidates": sorted(EVAL_EXCLUDE_CANDIDATES),
    "init_errors": init_errors,
    "candidates": {},
}

for name, predictor_base in registry.items():
    if name in EVAL_EXCLUDE_CANDIDATES:
        continue

    predictor_pre = predictor_base if name in NO_PREPROCESS_CANDIDATES else wrap_predictor_with_preprocess(predictor_base, mode=PREPROCESS_MODE)
    predictor = wrap_predictor_with_max_digits(predictor_pre, max_digits=MAX_READING_DIGITS)
    effective_preprocess_mode = "none" if name in NO_PREPROCESS_CANDIDATES else PREPROCESS_MODE

    poly = evaluate_predictor(predictor, wm_poly_test)
    bbox = evaluate_predictor(predictor, wm_bbox_test)
    pass_stage_a = (
        poly["non_empty_rate"] >= MIN_NON_EMPTY_RATE
        and bbox["non_empty_rate"] >= MIN_NON_EMPTY_RATE
        and poly["avg_ms"] <= MAX_AVG_MS
        and bbox["avg_ms"] <= MAX_AVG_MS
    )

    report["candidates"][name] = {
        "wm_polygon": poly,
        "wm_bbox": bbox,
        "effective_preprocess_mode": effective_preprocess_mode,
        "stage_a_pass": bool(pass_stage_a),
    }

    rows.append(
        {
            "candidate": name,
            "preprocess_mode": effective_preprocess_mode,
            "stage_a_pass": pass_stage_a,
            "poly_fsa_norm": poly["fsa_norm"],
            "bbox_fsa_norm": bbox["fsa_norm"],
            "poly_ms": poly["avg_ms"],
            "bbox_ms": bbox["avg_ms"],
        }
    )

    if pass_stage_a:
        method_suffix = f"_{effective_preprocess_mode}" if effective_preprocess_mode != "none" else ""
        row = build_ocr_comparison_row(
            method=f"pretrained_{name}{method_suffix}",
            wm_poly_metrics={"fsa_raw": poly["fsa_raw"], "fsa_norm": poly["fsa_norm"], "pda": poly["pda"], "cer": poly["cer"]},
            wm_bbox_metrics={"fsa_raw": bbox["fsa_raw"], "fsa_norm": bbox["fsa_norm"], "pda": bbox["pda"], "cer": bbox["cer"]},
            wm_poly_ms=poly["avg_ms"],
            wm_bbox_ms=bbox["avg_ms"],
            run_date=datetime.now().strftime("%Y-%m-%d %H:%M"),
        )
        append_ocr_comparison_row(RESULTS_DIR / "ocr_comparison.csv", row)

with open(RESULTS_DIR / "ocr_pretrained_pilot.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

display(pd.DataFrame(rows))
print("Preprocess mode:", PREPROCESS_MODE)
print("Excluded in Stage A eval:", sorted(EVAL_EXCLUDE_CANDIDATES))
print("Saved:", RESULTS_DIR / "ocr_pretrained_pilot.json")

Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alike\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alike\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alike\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


,candidate,preprocess_mode,stage_a_pass,poly_fsa_norm,bbox_fsa_norm,poly_ms,bbox_ms
0,ultralytics_yolo11m_baseline,none,True,0.754011,0.858289,57.825347,55.743279


Preprocess mode: gray_clahe
Excluded in Stage A eval: ['doctr', 'easyocr']
Saved: C:\Users\alike\WaterMeterCV\results\ocr_pretrained_pilot.json


In [50]:
import matplotlib.pyplot as plt

VIS_EXCLUDE_CANDIDATES = {"doctr", "easyocr"}
VIS_N = 60
VIS_DPI_DEFAULT = 140
VIS_DPI_ULTRALYTICS = 220


def _normalize_label(v: str) -> str:
    s = str(v)
    t = s.lstrip("0")
    return t if t else "0"


def _predict_ultralytics_digit_boxes(candidate_name: str, image_bgr):
    if candidate_name != "ultralytics_yolo11m_baseline":
        return []

    detections = extract_ultralytics_digit_detections(image_bgr)
    if not detections:
        return []

    detections, _ = apply_ultralytics_overlap_heuristic(detections)
    detections, _ = apply_ultralytics_last_drum_heuristic(detections)
    return detections


def _draw_error_bboxes(image_bgr, detections):
    vis = image_bgr.copy()
    h, w = vis.shape[:2]

    for det in detections:
        x1 = int(round(det["x1"]))
        y1 = int(round(det["y1"]))
        x2 = int(round(det["x2"]))
        y2 = int(round(det["y2"]))
        x1 = max(0, min(w - 1, x1))
        y1 = max(0, min(h - 1, y1))
        x2 = max(0, min(w - 1, x2))
        y2 = max(0, min(h - 1, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 1)

    return vis


def visualize_candidate(
    candidate_name: str,
    predictor,
    n: int = VIS_N,
    preprocess_mode: str = PREPROCESS_MODE,
    apply_preprocess: bool = True,
):
    fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 6.6), squeeze=False)

    rows = [
        ("wm_polygon", wm_poly_test),
        ("wm_bbox", wm_bbox_test),
    ]

    for row_i, (crop_key, samples) in enumerate(rows):
        for col_i in range(n):
            ax = axes[row_i, col_i]
            if col_i >= len(samples):
                ax.axis("off")
                continue

            img_path, gt = samples[col_i]
            image = cv2.imread(str(img_path))
            if image is None:
                ax.axis("off")
                continue

            dual = dual_read_inference(image, predictor)
            selected = select_dual_orientation_with_priors(dual, image_bgr=image)
            pred = selected.get("selected_pred", "")
            selected_angle = int(selected.get("selected_angle", 0))

            display_image = cv2.rotate(image, cv2.ROTATE_180) if selected_angle == 180 else image
            preview_img = preprocess_for_ocr(display_image, mode=preprocess_mode) if apply_preprocess else display_image.copy()

            gt_norm = _normalize_label(gt)
            pred_norm = _normalize_label(pred)
            ok_norm = pred_norm == gt_norm
            title_color = "green" if ok_norm else "red"

            if not ok_norm:
                error_boxes = _predict_ultralytics_digit_boxes(candidate_name, display_image)
                if error_boxes:
                    preview_img = _draw_error_bboxes(preview_img, error_boxes)

            sample_id = img_path.stem
            shown_pred = pred if pred else "EMPTY"
            ax.imshow(cv2.cvtColor(preview_img, cv2.COLOR_BGR2RGB))
            ax.set_title(f"ID={sample_id}\nGT={gt} | P={shown_pred}", fontsize=8, pad=2, color=title_color)
            if col_i == 0:
                ax.set_ylabel(crop_key, fontsize=9, fontweight="bold")
            ax.axis("off")

    effective_mode = preprocess_mode if apply_preprocess else "none"
    fig.suptitle(
        f"Pretrained OCR preview: {candidate_name} | preprocess={effective_mode}",
        fontsize=13,
    )
    fig.tight_layout(rect=[0.0, 0.03, 1.0, 0.93])

    out_path = RESULTS_DIR / f"ocr_pretrained_preview_{candidate_name}_{effective_mode}.png"
    save_dpi = VIS_DPI_ULTRALYTICS if candidate_name == "ultralytics_yolo11m_baseline" else VIS_DPI_DEFAULT
    fig.savefig(out_path, dpi=save_dpi)
    plt.close(fig)
    print(f"Saved preview: {out_path} (dpi={save_dpi})")


if "registry" not in globals() or not isinstance(registry, dict) or len(registry) == 0:
    registry, init_errors = build_registry()

if "init_errors" not in globals() or not isinstance(init_errors, dict):
    init_errors = {}

if "NO_PREPROCESS_CANDIDATES" not in globals():
    NO_PREPROCESS_CANDIDATES = {"ultralytics_yolo11m_baseline"}

if init_errors:
    print("Init errors:")
    for name, err in init_errors.items():
        print(f"  - {name}: {err}")

available_candidates = [name for name in registry.keys() if name not in VIS_EXCLUDE_CANDIDATES]

if not available_candidates:
    print("No available pretrained candidates to visualize.")
else:
    print(f"Visualizing {len(available_candidates)} candidates, n={VIS_N} examples each")
    print(f"Excluded candidates: {sorted(VIS_EXCLUDE_CANDIDATES)}")

    for chosen in available_candidates:
        apply_preprocess = chosen not in NO_PREPROCESS_CANDIDATES
        effective_mode = PREPROCESS_MODE if apply_preprocess else "none"
        predictor_for_vis_pre = (
            registry[chosen]
            if chosen in NO_PREPROCESS_CANDIDATES
            else wrap_predictor_with_preprocess(registry[chosen], mode=PREPROCESS_MODE)
        )
        predictor_for_vis = wrap_predictor_with_max_digits(predictor_for_vis_pre, max_digits=MAX_READING_DIGITS)

        print(f"Visualizing candidate: {chosen} | preprocess={effective_mode}")
        visualize_candidate(
            chosen,
            predictor_for_vis,
            n=VIS_N,
            preprocess_mode=PREPROCESS_MODE,
            apply_preprocess=apply_preprocess,
        )

Init errors:
  - paddleocr: runtime predict failed: NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at ..\paddle\fluid\framework\new_executor\instruction\onednn\onednn_instruction.cc:118)

  - pytesseract: tesseract is not installed or it's not in your PATH. See README file for more information.
Visualizing 1 candidates, n=60 examples each
Excluded candidates: ['doctr', 'easyocr']
Visualizing candidate: ultralytics_yolo11m_baseline | preprocess=none
Saved preview: C:\Users\alike\WaterMeterCV\results\ocr_pretrained_preview_ultralytics_yolo11m_baseline_none.png (dpi=220)


## Stage A+ Combination Sweep (Metrics + Failure Dumps)

Run extended comparison for available Stage A candidates on both crop paths (`wm_polygon`, `wm_bbox`), compute additional metrics, and save **only failed recognitions** to a dedicated folder.

In [67]:
if "VIS_EXCLUDE_CANDIDATES" not in globals():
    VIS_EXCLUDE_CANDIDATES = {"doctr", "easyocr"}

In [68]:
import shutil
from pathlib import Path

from models.metrics.evaluation import character_error_rate, per_digit_accuracy

COMBO_FAILURE_COMPARE_MODE = "normalized"  # "raw" or "normalized"
COMBO_FAILURES_ROOT = RESULTS_DIR / "ocr_pretrained_failures"
COMBO_FAILURES_ROOT.mkdir(parents=True, exist_ok=True)


def _normalize_digits_label(v: str) -> str:
    s = str(v)
    t = s.lstrip("0")
    return t if t else "0"


def _safe_token(v: str) -> str:
    token = digits_only(v)
    return token if token else "EMPTY"


def _is_ok_by_mode(pred: str, gt: str, mode: str) -> bool:
    if mode == "raw":
        return str(pred) == str(gt)
    if mode == "normalized":
        return _normalize_digits_label(pred) == _normalize_digits_label(gt)
    raise ValueError(f"Unsupported compare mode: {mode!r}")


if "_predict_ultralytics_digit_boxes" not in globals():
    def _predict_ultralytics_digit_boxes(candidate_name: str, image_bgr):
        return []

if "_draw_error_bboxes" not in globals():
    def _draw_error_bboxes(image_bgr, detections):
        return image_bgr


def _build_failure_preview(image_bgr, selected_angle, candidate_name, preprocess_mode, apply_preprocess):
    display_image = cv2.rotate(image_bgr, cv2.ROTATE_180) if selected_angle == 180 else image_bgr
    preview = (
        preprocess_for_ocr(display_image, mode=preprocess_mode)
        if apply_preprocess
        else display_image.copy()
    )

    error_boxes = _predict_ultralytics_digit_boxes(candidate_name, display_image)
    if error_boxes:
        preview = _draw_error_bboxes(preview, error_boxes)

    return preview


def evaluate_combo_and_dump_failures(
    candidate_name,
    predictor,
    samples,
    crop_key,
    preprocess_mode,
    apply_preprocess,
    compare_mode=COMBO_FAILURE_COMPARE_MODE,
):
    effective_mode = preprocess_mode if apply_preprocess else "none"
    combo_dir = COMBO_FAILURES_ROOT / candidate_name / effective_mode / crop_key

    if combo_dir.exists():
        shutil.rmtree(combo_dir)
    combo_dir.mkdir(parents=True, exist_ok=True)

    records = []
    failures = []

    for idx, (img_path, gt) in enumerate(samples):
        image = cv2.imread(str(img_path))
        if image is None:
            continue

        t0 = time.perf_counter()
        dual = dual_read_inference(image, predictor)
        selected = select_dual_orientation_with_priors(dual, image_bgr=image)
        dt_ms = (time.perf_counter() - t0) * 1000.0

        pred = str(selected.get("selected_pred", ""))
        angle = int(selected.get("selected_angle", 0))

        ok_raw = pred == str(gt)
        ok_norm = _normalize_digits_label(pred) == _normalize_digits_label(gt)
        ok_cmp = _is_ok_by_mode(pred, str(gt), compare_mode)

        rec = {
            "idx": int(idx),
            "sample_id": img_path.stem,
            "crop_key": crop_key,
            "candidate": candidate_name,
            "preprocess_mode": effective_mode,
            "gt": str(gt),
            "pred": pred,
            "gt_norm": _normalize_digits_label(str(gt)),
            "pred_norm": _normalize_digits_label(pred),
            "ok_raw": bool(ok_raw),
            "ok_norm": bool(ok_norm),
            "ok_cmp": bool(ok_cmp),
            "selected_angle": int(angle),
            "dt_ms": float(dt_ms),
            "pda_i": float(per_digit_accuracy(pred, str(gt))),
            "cer_i": float(character_error_rate(pred, str(gt))),
        }
        records.append(rec)

        if not ok_cmp:
            preview = _build_failure_preview(
                image,
                selected_angle=angle,
                candidate_name=candidate_name,
                preprocess_mode=preprocess_mode,
                apply_preprocess=apply_preprocess,
            )

            gt_token = _safe_token(str(gt))
            pred_token = _safe_token(pred)
            out_name = f"{idx:04d}__{img_path.stem}__gt_{gt_token}__pred_{pred_token}.png"
            out_path = combo_dir / out_name

            header = f"ID={img_path.stem} GT={gt} P={pred if pred else 'EMPTY'}"
            footer = f"mode={compare_mode} angle={angle} raw_ok={ok_raw} norm_ok={ok_norm}"
            cv2.putText(preview, header[:120], (4, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 0), 1, cv2.LINE_AA)
            cv2.putText(preview, footer[:120], (4, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 0), 1, cv2.LINE_AA)

            cv2.imwrite(str(out_path), preview)
            failures.append({**rec, "failure_image": str(out_path)})

    if not records:
        return None, None, None

    df = pd.DataFrame(records)
    failures_df = pd.DataFrame(failures)

    summary = {
        "candidate": candidate_name,
        "crop_key": crop_key,
        "compare_mode": compare_mode,
        "preprocess_mode": effective_mode,
        "samples": int(len(df)),
        "fsa_raw": float(df["ok_raw"].mean()),
        "fsa_norm": float(df["ok_norm"].mean()),
        "pda_mean": float(df["pda_i"].mean()),
        "cer_mean": float(df["cer_i"].mean()),
        "avg_ms": float(df["dt_ms"].mean()),
        "median_ms": float(df["dt_ms"].median()),
        "p95_ms": float(df["dt_ms"].quantile(0.95)),
        "non_empty_rate": float((df["pred"] != "").mean()),
        "selected_180_share": float((df["selected_angle"] == 180).mean()),
        "errors_saved": int((~df["ok_cmp"]).sum()),
        "failure_dir": str(combo_dir),
    }

    df.to_csv(combo_dir / "all_predictions.csv", index=False, encoding="utf-8")
    if not failures_df.empty:
        failures_df.to_csv(combo_dir / "failures.csv", index=False, encoding="utf-8")

    return summary, df, failures_df


if "registry" not in globals() or not isinstance(registry, dict) or len(registry) == 0:
    registry, init_errors = build_registry()

if "NO_PREPROCESS_CANDIDATES" not in globals():
    NO_PREPROCESS_CANDIDATES = {"ultralytics_yolo11m_baseline"}

combo_summaries = []
combo_records = []

available_candidates_combo = [name for name in registry.keys() if name not in VIS_EXCLUDE_CANDIDATES]
print(f"Combination sweep candidates: {available_candidates_combo}")
print(f"Failure compare mode: {COMBO_FAILURE_COMPARE_MODE}")

for chosen in available_candidates_combo:
    apply_preprocess = chosen not in NO_PREPROCESS_CANDIDATES
    effective_mode = PREPROCESS_MODE if apply_preprocess else "none"

    predictor_pre = (
        registry[chosen]
        if chosen in NO_PREPROCESS_CANDIDATES
        else wrap_predictor_with_preprocess(registry[chosen], mode=PREPROCESS_MODE)
    )
    predictor_combo = wrap_predictor_with_max_digits(predictor_pre, max_digits=MAX_READING_DIGITS)

    for crop_key, samples in [("wm_polygon", wm_poly_test), ("wm_bbox", wm_bbox_test)]:
        summary, df_rec, _ = evaluate_combo_and_dump_failures(
            candidate_name=chosen,
            predictor=predictor_combo,
            samples=samples,
            crop_key=crop_key,
            preprocess_mode=PREPROCESS_MODE,
            apply_preprocess=apply_preprocess,
            compare_mode=COMBO_FAILURE_COMPARE_MODE,
        )

        if summary is None:
            continue

        combo_summaries.append(summary)
        combo_records.append(df_rec)
        print(
            f"[{chosen} | {crop_key}] fsa_norm={summary['fsa_norm']:.4f} "
            f"fsa_raw={summary['fsa_raw']:.4f} errors_saved={summary['errors_saved']}"
        )

combo_df = pd.DataFrame(combo_summaries)
if combo_df.empty:
    print("No combination results were produced.")
else:
    combo_df = combo_df.sort_values(
        ["fsa_norm", "fsa_raw", "cer_mean", "avg_ms"],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)

    combo_csv_path = RESULTS_DIR / "ocr_pretrained_combinations_metrics.csv"
    combo_df.to_csv(combo_csv_path, index=False, encoding="utf-8")
    display(combo_df)
    print(f"Saved combination metrics: {combo_csv_path}")
    print(f"Saved failure images root: {COMBO_FAILURES_ROOT}")

Combination sweep candidates: ['ultralytics_yolo11m_baseline']
Failure compare mode: normalized
[ultralytics_yolo11m_baseline | wm_polygon] fsa_norm=0.7540 fsa_raw=0.0027 errors_saved=92
[ultralytics_yolo11m_baseline | wm_bbox] fsa_norm=0.8583 fsa_raw=0.0000 errors_saved=53


,candidate,crop_key,compare_mode,preprocess_mode,samples,fsa_raw,fsa_norm,pda_mean,cer_mean,avg_ms,median_ms,p95_ms,non_empty_rate,selected_180_share,errors_saved,failure_dir
0,ultralytics_yolo11m_baseline,wm_bbox,normalized,none,374,0.000000,0.858289,0.060658,0.469391,57.695055,57.1406,61.35139,1.0,0.066845,53,C:\Users\alike\WaterMeterCV\results\ocr_pretra...
1,ultralytics_yolo11m_baseline,wm_polygon,normalized,none,374,0.002674,0.754011,0.062347,0.486841,58.298498,57.0420,65.24978,1.0,0.056150,92,C:\Users\alike\WaterMeterCV\results\ocr_pretra...


Saved combination metrics: C:\Users\alike\WaterMeterCV\results\ocr_pretrained_combinations_metrics.csv
Saved failure images root: C:\Users\alike\WaterMeterCV\results\ocr_pretrained_failures


In [60]:
if "combo_df" not in globals() or combo_df is None or (hasattr(combo_df, "empty") and combo_df.empty):
    print("No combination dataframe found. Run previous cell first.")
else:
    best_row = combo_df.iloc[0]
    print("=== Stage A+ Decision Snapshot ===")
    print(f"Best combo: {best_row['candidate']} | {best_row['crop_key']} | preprocess={best_row['preprocess_mode']}")
    print(f"FSA norm: {best_row['fsa_norm']:.4f}")
    print(f"FSA raw : {best_row['fsa_raw']:.4f}")
    print(f"CER mean: {best_row['cer_mean']:.4f}")
    print(f"Avg ms  : {best_row['avg_ms']:.2f}")
    print(f"Failures: {int(best_row['errors_saved'])}")
    print("")

    by_candidate = (
        combo_df
        .groupby("candidate", as_index=False)
        .agg(
            fsa_norm_mean=("fsa_norm", "mean"),
            fsa_raw_mean=("fsa_raw", "mean"),
            cer_mean=("cer_mean", "mean"),
            avg_ms=("avg_ms", "mean"),
            errors_total=("errors_saved", "sum"),
        )
        .sort_values(["fsa_norm_mean", "fsa_raw_mean", "cer_mean", "avg_ms"], ascending=[False, False, True, True])
        .reset_index(drop=True)
    )
    display(by_candidate)

    top_candidate = by_candidate.iloc[0]
    print("Recommendation:")
    print(
        f"- Keep focus on bbox/polygon + {top_candidate['candidate']} for the next run cycle."
    )
    print("- Defer additional OCR families until this combo saturates on failure analysis and metric uplift.")
    print("- Use failure dumps in results/ocr_pretrained_failures for targeted iterations.")

=== Stage A+ Decision Snapshot ===
Best combo: ultralytics_yolo11m_baseline | wm_bbox | preprocess=none
FSA norm: 0.8770
FSA raw : 0.0000
CER mean: 0.4594
Avg ms  : 35.89
Failures: 46



,candidate,fsa_norm_mean,fsa_raw_mean,cer_mean,avg_ms,errors_total
0,ultralytics_yolo11m_baseline,0.814171,0.005348,0.467472,36.569812,139


Recommendation:
- Keep focus on bbox/polygon + ultralytics_yolo11m_baseline for the next run cycle.
- Defer additional OCR families until this combo saturates on failure analysis and metric uplift.
- Use failure dumps in results/ocr_pretrained_failures for targeted iterations.


## Stage A Rule

A candidate passes Stage A if both crop paths provide non-empty predictions and acceptable latency.
Final practical decision (Stage B) is made in `04_comparison.ipynb`.